# Gemini Embeddings — OpenAI Notebook Equivalent

**Goal:** repeat the same embedding lessons and practicals from `02_openai_embeddings.ipynb`, using Gemini instead of OpenAI.

**Learning flow:** API key → Gemini client → one embedding → multiple embeddings → cosine similarity → sentence-pair comparison → semantic search → second query → best match → embedding dimensions → food-review CSV practical → save vectors.

> **Provider difference:** OpenAI's notebook uses LangChain's `OpenAIEmbeddings` methods (`embed_query` and `embed_documents`). This notebook uses Google's `google-genai` SDK, so we define a small `embed_texts()` helper to get one Gemini vector per text. The concepts and practicals remain the same.

This notebook covers embeddings and in-memory semantic search only. It does **not** create a vector database.

## 1. Load the Gemini API Key Safely

Create or update `.env` in your project root:

```text
GEMINI_API_KEY=your_key_here
```

Do not paste the key directly into the notebook or commit `.env` to Git.

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

# Look for .env from the current folder and parent folders.
load_dotenv()

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    raise EnvironmentError(
        "GEMINI_API_KEY is not set. Add it to your .env file, then rerun this cell."
    )

print("✅ Gemini API key found (value hidden)")

✅ Gemini API key found (value hidden)


## 2. Initialize the Gemini Client

The client sends requests to Google's Gemini API. We will use it for embeddings and a short connection test.

In [2]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

EMBEDDING_MODEL = "gemini-embedding-2"
GENERATION_MODEL = "gemini-3.8-flash"

print("✅ Gemini client created")
print("Embedding model:", EMBEDDING_MODEL)
print("Generation test model:", GENERATION_MODEL)

✅ Gemini client created
Embedding model: gemini-embedding-2
Generation test model: gemini-3.8-flash


## 3. Test the Gemini API

This is a small generation request to check that the key can call the selected generation model. It is separate from the embedding calls below.

In [3]:
response = client.models.generate_content(
    model=GENERATION_MODEL,
    contents="Say hello in one sentence."
)

print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello, I hope you are having a wonderful day!


## 4. Embed One Text

An embedding converts text into a numeric vector. In the OpenAI notebook this was `embeddings.embed_query(single_text)`; here we call Gemini's `embed_content()` API directly.

In [4]:
single_text = (
    "LangChain and RAG are amazing frameworks and projects to work on"
)

single_result = client.models.embed_content(
    model=EMBEDDING_MODEL,
    contents=single_text
)

single_embedding = single_result.embeddings[0].values

print(f"Input: {single_text}")
print(f"Vector dimensions: {len(single_embedding)}")
print(f"Sample values: {single_embedding[:5]}")

Input: LangChain and RAG are amazing frameworks and projects to work on
Vector dimensions: 3072
Sample values: [-0.0050806366, 0.009851166, 0.00840434, 0.026356446, 0.010325184]


In [5]:
import json
from pathlib import Path

# Save to a shared location inside E:\RAG\data
cache_dir = Path.cwd().parent / "data" / "shared_embedding_cache"
cache_dir.mkdir(parents=True, exist_ok=True)

embedding_artifact = {
    "model": EMBEDDING_MODEL,
    "text": single_text,
    "embedding": [float(value) for value in single_embedding],
}

cache_file = cache_dir / "single_gemini_embedding.json"

cache_file.write_text(
    json.dumps(embedding_artifact),
    encoding="utf-8",
)

print("Embedding saved successfully!")
print("File:", cache_file.resolve())
print("Dimensions:", len(single_embedding))

Embedding saved successfully!
File: E:\RAG\data\shared_embedding_cache\single_gemini_embedding.json
Dimensions: 3072


### What just happened?

- `embed_content()` sends the text to Gemini's embedding endpoint.
- `result.embeddings[0].values` extracts the numeric vector.
- `len(single_embedding)` reports the actual number of dimensions returned by the model. We inspect it rather than assuming it.

## 5. Inspect the Embedding

Let's inspect the vector's Python types and shape, just as we did in the OpenAI practical.

In [6]:
print("Type of embedding:", type(single_embedding))
print("Type of one value:", type(single_embedding[0]))
print("Number of dimensions:", len(single_embedding))

Type of embedding: <class 'list'>
Type of one value: <class 'float'>
Number of dimensions: 3072


## 6. Embed Multiple Texts

Google's SDK can aggregate plain text inputs if they are passed as a simple list. To keep a reliable one-input-to-one-vector mapping in this practical, we wrap each sentence in a separate `Content` object.

In [7]:
def embed_texts(texts, client, model=EMBEDDING_MODEL):
    """Return one Gemini embedding vector for each input text."""
    if not texts:
        return []

    clean_texts = [str(text) for text in texts]
    if any(not text.strip() for text in clean_texts):
        raise ValueError("Input texts must not be empty strings.")

    contents = [
        types.Content(
            parts=[types.Part.from_text(text=text)]
        )
        for text in clean_texts
    ]

    result = client.models.embed_content(
        model=model,
        contents=contents
    )

    vectors = [item.values for item in result.embeddings]

    if len(vectors) != len(clean_texts):
        raise RuntimeError(
            f"Expected {len(clean_texts)} embeddings, received {len(vectors)}."
        )

    return vectors

In [8]:
multiple_texts = [
    "Python is a programming language",
    "LangChain is a framework for LLM applications",
    "Embeddings convert text to numbers",
    "Vectors can be compared for similarity"
]

multiple_embeddings = embed_texts(multiple_texts, client)

print(f"Number of texts: {len(multiple_texts)}")
print(f"Number of embeddings: {len(multiple_embeddings)}")
print(f"Dimensions of first embedding: {len(multiple_embeddings[0])}")

Number of texts: 4
Number of embeddings: 4
Dimensions of first embedding: 3072


In [9]:
# Inspect the first vector (this can be a long list of numbers).
multiple_embeddings[0][:10]

[0.01145267,
 -0.0011555406,
 0.0068883724,
 0.0034775974,
 -0.0045854896,
 -0.003977132,
 -0.013933986,
 -0.008749574,
 0.026684884,
 -0.063887656]

## 7. Cosine Similarity

Cosine similarity compares the direction of two vectors.

\[
\text{cosine similarity}(A,B)=\frac{A\cdot B}{\|A\|\,\|B\|}
\]

We implement the math ourselves. The function name `cosine_similarity_manual` is intentional: it avoids accidentally calling scikit-learn's function, which expects 2D arrays.

In [10]:
import numpy as np


def cosine_similarity_manual(vec1, vec2):
    """Cosine similarity for two one-dimensional embedding vectors."""
    vec1 = np.asarray(vec1, dtype=float).reshape(-1)
    vec2 = np.asarray(vec2, dtype=float).reshape(-1)

    if vec1.shape != vec2.shape:
        raise ValueError(
            f"Vector dimensions must match; got {vec1.shape} and {vec2.shape}."
        )

    dot_product = np.dot(vec1, vec2)
    norm_a = np.linalg.norm(vec1)
    norm_b = np.linalg.norm(vec2)

    if norm_a == 0 or norm_b == 0:
        raise ValueError("Cosine similarity is undefined for a zero vector.")

    return float(dot_product / (norm_a * norm_b))

## 8. Compare All Sentence Pairs

These are the same example sentences used in the OpenAI notebook. There are five sentences, so the loop compares each unique pair once (10 pairs total).

In [11]:
sentences = [
    "The cat sat on the mat",
    "A feline rested on the rug",
    "The dog played in the yard",
    "I love programming in Python",
    "Python is my favorite programming language"
]

sentence_embeddings = embed_texts(sentences, client)

print("Number of sentences:", len(sentences))
print("Number of embeddings:", len(sentence_embeddings))
print("Vector dimensions:", len(sentence_embeddings[0]))

Number of sentences: 5
Number of embeddings: 5
Vector dimensions: 3072


In [12]:
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        similarity = cosine_similarity_manual(
            sentence_embeddings[i],
            sentence_embeddings[j]
        )

        print(f"'{sentences[i]}'")
        print(f"vs '{sentences[j]}'")
        print(f"Similarity: {similarity:.3f}")
        print("-" * 70)

'The cat sat on the mat'
vs 'A feline rested on the rug'
Similarity: 0.670
----------------------------------------------------------------------
'The cat sat on the mat'
vs 'The dog played in the yard'
Similarity: 0.579
----------------------------------------------------------------------
'The cat sat on the mat'
vs 'I love programming in Python'
Similarity: 0.573
----------------------------------------------------------------------
'The cat sat on the mat'
vs 'Python is my favorite programming language'
Similarity: 0.578
----------------------------------------------------------------------
'A feline rested on the rug'
vs 'The dog played in the yard'
Similarity: 0.468
----------------------------------------------------------------------
'A feline rested on the rug'
vs 'I love programming in Python'
Similarity: 0.401
----------------------------------------------------------------------
'A feline rested on the rug'
vs 'Python is my favorite programming language'
Similarity: 0.411
-

### What should we notice?

`The cat sat on the mat` and `A feline rested on the rug` have related meanings even though the words differ. Their similarity score should help demonstrate why embeddings support semantic search. Exact scores can vary by model and input.

## 9. Semantic Search

Semantic search ranks documents by meaning rather than exact keyword overlap. We'll use the same five documents as the OpenAI notebook.

In [13]:
documents = [
    "LangChain is a framework for developing applications powered by language models",
    "Python is a high-level programming language",
    "Machine learning is a subset of artificial intelligence",
    "Embeddings convert text into numerical vectors",
    "The weather today is sunny and warm"
]

# Create document vectors once. Reuse them for every query.
document_embeddings = embed_texts(documents, client)

print("Documents:", len(documents))
print("Document embeddings:", len(document_embeddings))
print("Vector dimensions:", len(document_embeddings[0]))

Documents: 5
Document embeddings: 5
Vector dimensions: 3072


In [14]:
def semantic_search(query, documents, document_embeddings, client, top_k=3):
    """Rank documents by cosine similarity to a Gemini query embedding."""
    if len(documents) != len(document_embeddings):
        raise ValueError("Each document must have exactly one embedding.")
    if not query.strip():
        raise ValueError("Query must not be empty.")
    if top_k < 1:
        raise ValueError("top_k must be at least 1.")
    if not documents:
        return []

    # Only the new query needs to be embedded at search time.
    query_embedding = embed_texts([query], client)[0]

    scored_documents = []
    for document, document_embedding in zip(documents, document_embeddings):
        score = cosine_similarity_manual(query_embedding, document_embedding)
        scored_documents.append((score, document))

    scored_documents.sort(key=lambda item: item[0], reverse=True)
    return scored_documents[:min(top_k, len(scored_documents))]

### Why precompute document embeddings?

For the first search, documents are embedded once. Later searches create only a new query embedding, then compare it with the saved in-memory vectors. This avoids repeating the same document-embedding API calls for every query.

## 10. Try Semantic Search

Ask: `What is LangChain?` The function returns the top three documents by similarity.

In [15]:
query = "What is LangChain?"

results = semantic_search(
    query,
    documents,
    document_embeddings,
    client,
    top_k=3
)

for score, document in results:
    print(f"Score: {score:.3f} | {document}")

Score: 0.877 | LangChain is a framework for developing applications powered by language models
Score: 0.650 | Machine learning is a subset of artificial intelligence
Score: 0.645 | Python is a high-level programming language


## 11. Try a Second Query

Change the question and observe how the ranking changes. The document embeddings are reused; only the query is embedded again.

In [16]:
query = "What are embeddings?"

results = semantic_search(
    query,
    documents,
    document_embeddings,
    client,
    top_k=3
)

print(f"\nSemantic Search Results for: '{query}'\n")
for score, document in results:
    print(f"Score: {score:.3f} | {document}")


Semantic Search Results for: 'What are embeddings?'

Score: 0.872 | Embeddings convert text into numerical vectors
Score: 0.644 | LangChain is a framework for developing applications powered by language models
Score: 0.638 | Machine learning is a subset of artificial intelligence


## 12. Retrieve the Best Document Only

Sometimes we only need the single highest-scoring document.

In [17]:
if results:
    best_score, best_document = results[0]
    print(f"Best similarity score: {best_score:.3f}")
    print(f"Best matching document: {best_document}")
else:
    print("No results found.")

Best similarity score: 0.872
Best matching document: Embeddings convert text into numerical vectors


## 13. Gemini Embedding Dimensions

The OpenAI notebook compared different embedding models. For this Gemini practical, we will compare supported output sizes from the same `gemini-embedding-2` model. Google's documentation lists 768, 1536, and 3072 as recommended output dimensions; the default is 3072.

This optional cell makes three API calls. Run it when you want to see how output dimensionality changes vector size.

In [18]:
dimension_examples = {}

for dimension in [768, 1536, 3072]:
    result = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=single_text,
        config=types.EmbedContentConfig(output_dimensionality=dimension)
    )
    vector = result.embeddings[0].values
    dimension_examples[dimension] = vector
    print(
        f"Requested dimensions: {dimension} | Returned dimensions: {len(vector)}")

Requested dimensions: 768 | Returned dimensions: 768
Requested dimensions: 1536 | Returned dimensions: 1536
Requested dimensions: 3072 | Returned dimensions: 3072


## 14. Real Dataset Practical — Food Reviews CSV

This is an additional practical for the dataset section you've been working on.

The notebook will:
1. create a local `data` folder if needed;
2. try to download the OpenAI Cookbook sample CSV if it isn't already present;
3. use a small built-in sample if the download fails, so `df` is still defined and the notebook can continue;
4. select the review columns and combine summary + full text.

**Important:** the dataset is only sample text. All embeddings in this notebook are generated with Gemini. No vector database is created.

### Why do we create `combined`?

A short summary may not contain enough detail on its own. We combine the title/summary and the review body into one text field so an embedding can represent more of the review's meaning.

In [19]:
from pathlib import Path
import urllib.request
import pandas as pd

DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

input_datapath = DATA_DIR / "fine_food_reviews_1k.csv"
dataset_url = (
    "https://raw.githubusercontent.com/openai/openai-cookbook/"
    "main/examples/data/fine_food_reviews_1k.csv"
)

# Small fallback for offline/restricted environments.
demo_reviews = [
    {"Time": 1, "ProductId": "DEMO001", "UserId": "USER001", "Score": 5,
     "Summary": "Great taste", "Text": "Fresh and tasty. I would buy it again."},
    {"Time": 2, "ProductId": "DEMO002", "UserId": "USER002", "Score": 4,
     "Summary": "Good value", "Text": "Good quality for the price."},
    {"Time": 3, "ProductId": "DEMO003", "UserId": "USER003", "Score": 2,
     "Summary": "Too sweet", "Text": "The product was much sweeter than expected."},
    {"Time": 4, "ProductId": "DEMO004", "UserId": "USER004", "Score": 5,
     "Summary": "Loved it", "Text": "Excellent flavour and nice packaging."},
    {"Time": 5, "ProductId": "DEMO005", "UserId": "USER005", "Score": 3,
     "Summary": "Average product", "Text": "It was okay, but nothing special."},
    {"Time": 6, "ProductId": "DEMO006", "UserId": "USER006", "Score": 1,
     "Summary": "Poor quality", "Text": "The package arrived damaged."},
    {"Time": 7, "ProductId": "DEMO007", "UserId": "USER007", "Score": 4,
     "Summary": "Nice snack", "Text": "A convenient snack with a pleasant taste."},
    {"Time": 8, "ProductId": "DEMO008", "UserId": "USER008", "Score": 5,
     "Summary": "Worth buying", "Text": "Very good and better than similar products."},
    {"Time": 9, "ProductId": "DEMO009", "UserId": "USER009", "Score": 2,
     "Summary": "Not fresh", "Text": "It did not taste fresh, so I would not order again."},
    {"Time": 10, "ProductId": "DEMO010", "UserId": "USER010", "Score": 4,
     "Summary": "Pretty good", "Text": "Good flavour and an acceptable texture."},
]

data_source = None

if not input_datapath.exists():
    print(f"Dataset not found locally: {input_datapath.resolve()}")
    print("Trying to download the sample CSV...")
    try:
        request = urllib.request.Request(
            dataset_url,
            headers={"User-Agent": "Gemini-Embeddings-Learning-Notebook"}
        )
        with urllib.request.urlopen(request, timeout=20) as response:
            csv_bytes = response.read()
        if not csv_bytes:
            raise ValueError("Downloaded CSV was empty.")
        input_datapath.write_bytes(csv_bytes)
        print(f"✅ Downloaded CSV: {input_datapath.resolve()}")
    except Exception as download_error:
        print(
            f"⚠️ Download failed: {type(download_error).__name__}: {download_error}")
        print("Using the built-in 10-row demo dataset instead.")
        df = pd.DataFrame(demo_reviews)
        data_source = "built-in demo dataset"

if data_source is None:
    df = pd.read_csv(input_datapath)
    # Remove a saved pandas index column if the file contains one.
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed:")]
    data_source = "local/downloaded food-review CSV"

required_columns = ["Time", "ProductId", "UserId", "Score", "Summary", "Text"]
missing_columns = [
    column for column in required_columns if column not in df.columns]
if missing_columns:
    raise ValueError(
        f"Dataset is missing columns {missing_columns}. Found: {list(df.columns)}"
    )

df = df[required_columns].copy()
df["Summary"] = df["Summary"].fillna("").astype(str).str.strip()
df["Text"] = df["Text"].fillna("").astype(str).str.strip()
df = df[(df["Summary"] + df["Text"]).str.strip().ne("")].copy()

df["combined"] = (
    "Title: " + df["Summary"] + "; Content: " + df["Text"]
)

if df.empty:
    raise ValueError("No non-empty reviews were found in the dataset.")

print("✅ Dataset ready")
print("Source:", data_source)
print("Rows:", len(df))
df.head(2)

✅ Dataset ready
Source: local/downloaded food-review CSV
Rows: 1000


,Time,ProductId,UserId,Score,Summary,Text,combined
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...


## 15. Generate Gemini Embeddings for a Small Batch

Start with 10 reviews to keep the first API experiment small. If the downloaded dataset is unavailable, the fallback still gives us 10 demo reviews.

In [20]:
sample_df = df.head(10).copy()
sample_texts = sample_df["combined"].tolist()

sample_embeddings = embed_texts(sample_texts, client)

print("Documents:", len(sample_texts))
print("Embeddings:", len(sample_embeddings))
print("Vector dimensions:", len(sample_embeddings[0]))

Documents: 10
Embeddings: 10
Vector dimensions: 3072


## 16. Attach Embeddings to the DataFrame

Each row now contains the review metadata, combined text, and its Gemini embedding vector.

In [21]:
sample_df["embedding"] = sample_embeddings

sample_df[["ProductId", "Score", "Summary", "combined", "embedding"]].head(2)

,ProductId,Score,Summary,combined,embedding
0,B003XPF9BO,5,where does one start...and stop... with a tre...,Title: where does one start...and stop... wit...,"[-0.010978692, 0.0037329271, -0.027429918, -0...."
1,B003JK537S,1,Arrived in pieces,Title: Arrived in pieces; Content: Not pleased...,"[-0.0011592726, 0.010573441, -0.0098881945, 0...."


## 17. Save the Embeddings

Save the 10-row learning sample to CSV. This is a simple file export—not a vector database.

In [22]:
output_path = DATA_DIR / "fine_food_reviews_gemini_embeddings_10.csv"
sample_df.to_csv(output_path, index=False)

print(f"✅ Saved embeddings CSV: {output_path.resolve()}")

✅ Saved embeddings CSV: E:\RAG\Vector_embeddings_&_Databases\data\fine_food_reviews_gemini_embeddings_10.csv


## 18. Production Insight

### Learning version

```text
Query → embed query → compare with document vectors → top-K results
```

### Production-style architecture

```text
INGESTION
Documents → embeddings → persist/index vectors

QUERY
User query → query embedding → vector search → top-K context
```

For this notebook we stop at in-memory similarity search and CSV export. A vector database such as Chroma can be a later lesson, but it is not built here.

## 19. How This Relates to RAG

This notebook demonstrates the embedding and retrieval layer used in RAG:

```text
Documents → Gemini embeddings → similarity search → top-K retrieved text
                                                  ↓
                                         Gemini generation model
                                                  ↓
                                              Answer
```

Embedding and generation are separate components even when both are provided by Gemini.

## 20. Final Takeaways

- Embeddings convert text into numerical vectors.
- `gemini-embedding-2` is the embedding model used in this practical.
- `embed_texts()` keeps one input text mapped to one embedding vector.
- The custom `cosine_similarity_manual()` function avoids confusion with scikit-learn's 2D-input function.
- Semantic search ranks documents by similarity to a query.
- Document embeddings are precomputed once for repeated queries.
- The CSV section downloads the sample when possible and uses a built-in fallback when offline.
- No vector database is created in this notebook.

## 21. Optional: Check the Installed SDK

Run this only when debugging the environment:

```bash
uv pip show google-genai
```

If the package is missing:

```bash
uv add google-genai python-dotenv numpy pandas
```